# LunaQuest / Lunar IceNav Research Workflow

Research notebook for BAH 2026 Problem Statement 8. The workflow produces radar-based candidate patch screening, candidate patch scientific review, preliminary landing candidates, conceptual rover routes, weak ML pseudo-label experiments, and validation reports. Outputs require independent validation.

## A. Problem and Research Objective

Goal: detect radar-based candidate subsurface ice regions, estimate candidate patch extent, identify preliminary safe landing candidates, plan conceptual rover navigation paths, and evaluate model reliability with proxy/scientific consistency metrics.

In [ ]:
from pathlib import Path
import sys

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'configs' / 'pipeline.json').exists() and (candidate / 'src').exists():
            return candidate
    raise RuntimeError('Could not find project root containing configs/pipeline.json and src/')

root = find_project_root(Path.cwd().resolve())
sys.path.insert(0, str(root / 'src'))
figdir = root / 'outputs' / 'figures'
tables = root / 'outputs' / 'tables'
routes_dir = root / 'outputs' / 'routes'
print(root)

## B. Region Validation and AOI

Configured AOI: lat -87.8 to -86.9, lon 80 to 85 E. Primary SAR selection is based on full AOI coverage, while partial/no-overlap products are supporting or excluded.

In [ ]:
import pandas as pd
from IPython.display import display, Image
import json
manifest = json.loads((root / 'reports' / 'run_manifest.json').read_text())
print('AOI:', manifest['aoi'])
display(pd.read_csv(tables / 'sar_product_selection_reason.csv')[['product_id','coverage_fraction','selected_for_main_map','selection_reason']])
display(Image(filename=str(figdir / 'region_validation_map.png')))

## C. Data Inventory and Product Selection

Inventory and coverage tables document what was available, what was selected, and what remains context-only.

In [ ]:
inv = pd.read_csv(tables / 'product_inventory.csv')
display(inv['product_type'].value_counts().reset_index(name='rows'))
display(pd.read_csv(tables / 'data_coverage_status.csv'))
display(Image(filename=str(figdir / '01_dataset_inventory_chart.png')))
display(Image(filename=str(figdir / '02_selected_sar_coverage.png')))

## D. SAR Feature Extraction

Features include SAR log intensity, LH/LV ratio proxy, LV/LH ratio proxy, polarization imbalance proxy, local mean, local standard deviation, texture roughness, candidate score, and threshold uncertainty. True CPR/DOP are not claimed.

In [ ]:
sar_meta = pd.read_csv(tables / 'selected_sar_metadata.csv')
display(sar_meta.T)
features = __import__('numpy').load(root / 'outputs' / 'features' / 'sar_feature_stack.npz')
print('Feature arrays:', list(features.keys()))
display(Image(filename=str(figdir / '04_sar_feature_panel.png')))

## E. Candidate Mask Generation

The candidate map is generated before landing analysis. It uses SAR proxy thresholds, connected components, and threshold sensitivity checks.

In [ ]:
candidate_summary = pd.read_csv(tables / 'candidate_patch_summary.csv')
thresh = pd.read_csv(tables / 'threshold_sensitivity.csv')
display(candidate_summary)
display(thresh)
display(Image(filename=str(figdir / 'ice_candidate_detection_map.png')))
display(Image(filename=str(figdir / 'threshold_sensitivity_curve.png')))

## F. Candidate Patch Scientific Review

Each candidate patch is evaluated for extent, equivalent candidate patch diameter, score, ratio proxy, texture, slope context, threshold stability, uncertainty, confidence, landing proximity, and route accessibility.

In [ ]:
review = pd.read_csv(tables / 'candidate_scientific_review.csv')
display(review.head(12))
for name in ['ice_candidate_confidence_map.png','ice_candidate_ranking_chart.png','candidate_area_vs_score_scatter.png','candidate_diameter_distribution.png','candidate_score_vs_slope.png']:
    display(Image(filename=str(figdir / name)))

## G. Candidate Patch Extent and Resource Scenario Estimation

Equivalent candidate patch diameter is an area-derived screening extent. Scenario-based resource volume is a planning case only if a patch is later validated.

In [ ]:
resource = pd.read_csv(tables / 'resource_scenario_estimates.csv')
display(resource.head(18))
display(Image(filename=str(figdir / 'resource_scenario_bar_chart.png')))

## H. DEM/Slope Terrain Safety

DEM-derived slope classes support landing and traverse screening. They are not a complete hazard analysis.

In [ ]:
slope_summary = pd.read_csv(tables / 'slope_safety_summary.csv')
display(slope_summary)
for name in ['07_dem_elevation_map.png','08_dem_slope_map.png','08b_slope_classification_map.png']:
    display(Image(filename=str(figdir / name)))

## I. Landing Site Model

Landing scoring uses the candidate map as input, then searches for preliminary landing candidates near candidate patches while avoiding the candidate mask and risky/steep terrain.

In [ ]:
landing = pd.read_csv(tables / 'landing_site_evaluation.csv')
display(landing)
for name in ['09_landing_suitability_map.png','landing_score_components.png','landing_candidate_decision_map.png','nearest_landing_to_candidate_map.png']:
    display(Image(filename=str(figdir / name)))

## J. Rover Navigation Model

The rover planner generates shortest, safest, science-priority, and energy-efficient conceptual route variants, then evaluates slope exposure, risk proxy, science reward, and energy proxy.

In [ ]:
rover = pd.read_csv(tables / 'rover_navigation_evaluation.csv')
display(rover[['route_type','target_candidate_id','start_landing_site_id','length_m','total_cost','percent_route_under_5deg','percent_route_above_10deg','science_reward_score','energy_cost_proxy','traverse_risk_score']])
for name in ['rover_route_decision_map.png','rover_route_slope_profile.png','rover_route_risk_profile.png']:
    display(Image(filename=str(figdir / name)))

## K. U-Net Weakly Supervised Experiment

The U-Net and baselines are evaluated against rule-based pseudo-labels only. Metrics are pseudo-label agreement, not independently validated ice detection accuracy.

In [ ]:
metrics = pd.read_csv(tables / 'unet_pseudo_label_metrics.csv')
experiments = pd.read_csv(tables / 'model_experiment_comparison.csv')
display(metrics.T)
display(experiments)
for name in ['unet_training_curve.png','model_experiment_comparison.png','unet_prediction_overlay.png','unet_error_map_against_pseudolabel.png']:
    display(Image(filename=str(figdir / name)))

## L. Model Evaluation Summary

This section links the candidate map, patch confidence, landing dependency, rover navigation, and weak ML experiments into one validation-oriented summary.

In [ ]:
print((root / 'reports' / 'MODEL_EVALUATION_REPORT.md').read_text()[:5000])

## M. Technical Limitations

Key limitations include no direct water-ice confirmation claim, no true CPR/DOP claim yet, context-only OHRC, preliminary landing candidates, conceptual rover routes, and pseudo-label-only ML agreement.

In [ ]:
print((root / 'reports' / 'TECHNICAL_LIMITATIONS.md').read_text())

## N. Next Data to Download

The next data downloads should improve validation, hazard analysis, and physics-based interpretation.

In [ ]:
print((root / 'reports' / 'NEXT_DATA_TO_DOWNLOAD.md').read_text())